<a href="https://colab.research.google.com/github/arthurfariasaraujo/disciplinauniesp/blob/main/C%C3%B3pia_de_Aula_00_Dados_na_Pratica_com_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Aula 00 · Dados na Prática com IA
### Amazing Data IA Lab · Nerd Data Lab + UNIESP Centro Universitário

☕ **O caso:** o dono do **Café Tambaú**, uma cafeteria (fictícia) de João Pessoa, percebeu que o caixa anda mais vazio. Ele não sabe por quê, e contratou **você** como analista de dados.

> **Sua missão:** descobrir **por que as vendas caíram** e dizer ao dono o que fazer.

Para isso você vai percorrer o caminho de qualquer projeto de dados:

🧺 **Coletar** → 🧼 **Limpar** → 🔧 **Transformar** → 🔍 **Examinar** → 💡 **Insights** → ✅ **Decisão**

É assim que dados brutos viram informação útil: você coleta, limpa, transforma e examina os dados para descobrir **insights**, identificar **padrões** e **apoiar decisões**.

🤖 **Seu copiloto é a IA generativa** (ChatGPT, Gemini, Claude...). Ela escreve código, explica erros e sugere caminhos. **Você** pensa, decide e confere.

| Etapa | O que vamos fazer | Tempo |
|---|---|---|
| 🐍 Python | variáveis, listas e tabelas | 10 min |
| 🧺 Coletar | ler os arquivos do sistema do café | 10 min |
| 🧼 Limpar | consertar 5 problemas dos dados brutos | 30 min |
| 🔧 Transformar | juntar tabelas e criar colunas novas | 10 min |
| 🔍 Examinar | fazer perguntas aos dados com SQL | 25 min |
| 💡 Insights e ✅ Decisão | contar a história ao dono do café | 15 min |

## 🧭 Como usar este notebook

1. 💾 **Primeiro:** menu **Arquivo → Salvar uma cópia no Drive**. Assim as suas respostas ficam com você.
2. ▶️ Clique em uma célula e aperte **Shift + Enter** para rodar. Rode **uma por uma, em ordem**. Não use "Executar tudo".
3. ✏️ As células com `___` (três traços baixos) são **campos para completar**. Use sempre este ritual:

   **🧠 PENSE** (o que eu quero?) → **🤖 PERGUNTE** (se precisar, à IA) → **▶️ EXECUTE** → **✅ CONFIRA**

4. ✅ As células de **conferência** dizem se você acertou e dão uma dica se ainda não.
5. 🆘 Deu um erro estranho? Veja se sobrou algum `___` sem preencher. Se não, copie o erro e peça ajuda à IA (receita abaixo).
6. 📍 Travou mesmo? Há uma **saída de emergência** no final do notebook (Apêndice).

## 🧪 A receita de um bom prompt

Um prompt é o pedido que você faz à IA. Quanto melhor o pedido, melhor a resposta:

| Ingrediente | Exemplo |
|---|---|
| 🎭 **Quem você é** | "Sou iniciante em análise de dados e uso Python (pandas) no Google Colab." |
| 📋 **O que você quer** | "Quero transformar a coluna `preco_unitario` em número." |
| 🗂️ **Seus dados** | nomes das colunas + 3 a 5 linhas de exemplo (este notebook gera isso para você!) |
| 📦 **Formato da resposta** | "Responda em 3 partes: explicação simples, código pronto, como conferir." |
| 🚧 **Limites** | "Use só pandas. Não apague a tabela original." |

## 🏅 Regras de ouro ao trabalhar com IA

1. **Entenda antes de colar.** Peça: "explique linha por linha".
2. **Confira o resultado.** Conte as linhas antes e depois, olhe 5 exemplos.
3. **Deu erro?** Copie a mensagem de erro **inteira** e cole na IA junto com o seu código.
4. **A IA pode errar, e até inventar números.** Os números certos estão nos seus dados.
5. **Nunca cole dados reais de pessoas** (nome, CPF, telefone, e-mail) em chatbots. Aqui os dados são fictícios, então pode.

> 💡 Se o seu Colab mostrar o botão do **Gemini** (✨), você também pode pedir ajuda por lá, sem sair do notebook.

## 🧠 Antes de começar: qual é o seu palpite?

Por que você acha que as vendas de uma cafeteria caem de repente? Preço? Concorrente? Entrega? Clima? Algo que o dono fez?

✍️ **Clique duas vezes aqui e escreva o seu palpite. No final do notebook vamos conferir quem acertou.**

> *Meu palpite:* ...

In [8]:
# ⚙️ Configuração: rode esta célula primeiro (Shift + Enter)
import os
import pandas as pd
import matplotlib.pyplot as plt

try:
    import duckdb
except ImportError:
    %pip install -q duckdb
    import duckdb

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (9, 4)

# De onde vêm os dados (não precisa mexer):
URL_BASE = "https://raw.githubusercontent.com/wuldson-franco/workshop_dados_pratica/main/"

def ler(arquivo, **opcoes):
    """Lê um arquivo CSV: da pasta local, se existir; senão, baixa do GitHub."""
    if os.path.exists(arquivo):
        return pd.read_csv(arquivo, **opcoes)
    return pd.read_csv(URL_BASE + arquivo, **opcoes)

print("Configuração pronta! ✅")

Configuração pronta! ✅


In [7]:
# 🧰 Ferramentas do notebook (não precisa entender agora, só rode)

def conferir(condicao, ok, dica):
    """Mostra ✅ se você acertou ou ❌ com uma dica."""
    try:
        passou = bool(condicao)
    except Exception:
        passou = False
    if passou:
        print("✅", ok)
    else:
        print("❌ Ainda não:", dica)


def sql(consulta):
    """Roda uma consulta SQL sobre as tabelas (DataFrames) que existem no notebook."""
    resultado = duckdb.sql(consulta)
    return None if resultado is None else resultado.df()


def prompt_com_amostra(tabela, nome, pedido, linhas=5):
    """Monta um prompt completo (com colunas e exemplos dos seus dados) para colar na IA."""
    amostra = tabela.sample(linhas, random_state=7).to_string(index=False)
    return f"""Sou iniciante em análise de dados e uso Python (pandas) no Google Colab.

Minha tabela se chama `{nome}` e tem estas colunas:
{", ".join(tabela.columns)}

Algumas linhas de exemplo (dados fictícios):
{amostra}

O que eu preciso:
{pedido}

Responda em português, em 3 partes:
1) Explicação em linguagem simples (no máximo 4 linhas)
2) Código pronto para colar em uma célula do Colab (somente pandas)
3) Como eu confiro se deu certo
Não apague a tabela original."""


def prompt_sql(tabela, nome, pergunta, linhas=3):
    """Monta um prompt para a IA escrever SQL sobre a sua tabela."""
    colunas = "\n".join(f"- {c} ({t})" for c, t in tabela.dtypes.astype(str).items())
    amostra = tabela.head(linhas).to_string(index=False)
    return f"""Você é especialista em SQL (dialeto DuckDB) e está ajudando um iniciante.

Tabela `{nome}` (dados fictícios de uma cafeteria). Colunas e tipos:
{colunas}

Algumas linhas de exemplo:
{amostra}

Pergunta de negócio: {pergunta}

Responda em 3 partes:
1) O SQL pronto para copiar
2) Explicação de cada linha em português simples
3) Como conferir se o resultado faz sentido
Use somente a tabela `{nome}`."""


def prompt_insights(achados):
    """Monta um prompt para a IA ajudar a contar a história ao dono do café."""
    return f"""Você é analista de dados e vai explicar resultados ao dono de uma cafeteria (Café Tambaú, João Pessoa), que não conhece de dados.

A pergunta dele: por que as vendas caíram?

Meus achados, com números tirados dos dados:
{achados}

Escreva:
1) Um resumo de 3 frases, sem jargão.
2) 3 recomendações práticas, da mais urgente para a menos urgente.
3) Uma limitação: o que esses dados NÃO provam.

Regras: use SOMENTE os números que eu informei. Se faltar informação, diga o que falta em vez de inventar."""


print("Ferramentas prontas! ✅")

Ferramentas prontas! ✅


---
# 🐍 Parte 1 · Python

**Python** é a linguagem que vamos usar para conversar com o computador. Três ideias bastam para começar:

1. **Variáveis**: caixinhas com nome onde guardamos um valor.
2. **Listas**: várias coisas guardadas em uma só caixinha.
3. **Tabelas (DataFrames)**: como uma planilha do Excel, só que dentro do Python. É com elas que vamos trabalhar o tempo todo.

### 1.1 Variáveis
Texto vai **entre aspas**. Número decimal usa **ponto**, não vírgula (`9.90`, não `9,90`).

In [6]:
produto = "Cappuccino"      # texto (entre aspas)
preco = 9.90                # número com casas decimais
quantidade = 3              # número inteiro

total = round(preco * quantidade, 2)      # round(..., 2) arredonda para 2 casas
print("O pedido de", quantidade, produto, "custa R$", total)

O pedido de 3 Cappuccino custa R$ 29.7


🧠 **Pense:** o Cappuccino subiu para **R$ 12,90** e um cliente pediu **2**. Quanto custa o pedido? Escreva a conta no lugar do `___`.

In [12]:
# ✏️ COMPLETE: troque os _ pelo código certo
preco_novo = 12.90
quantidade_nova = 2

total_novo = 25.8        # dica: preço vezes quantidade, dentro de round(..., 2)
print("Novo total: R$", total_novo)

conferir(total_novo == 25.8, "Certo! 12,90 x 2 = 25,80.", "o total deveria ser 25.8. Confira a conta e se usou o ponto como separador.")

Novo total: R$ 25.8
✅ Certo! 12,90 x 2 = 25,80.


### 1.2 Listas
Uma lista guarda vários itens entre colchetes. **O Python começa a contar do zero**: o primeiro item é o `0`.

In [ ]:
cardapio = ["Café expresso", "Cappuccino", "Pão de queijo", "Tapioca"]

print("Itens no cardápio:", len(cardapio))
print("Primeiro item:", cardapio[0])

🧠 **Pense:** como **adicionar** "Brigadeiro" ao cardápio? E como pegar o **último** item da lista? (Dica: o último é o `-1`.)

> 🤖 **Dica de IA:** se não lembrar o nome do comando, pergunte: *"Como adiciono um item no fim de uma lista em Python? Dê um exemplo curto."*

In [15]:
cardapio = ["Café expresso", "Cappuccino", "Pão de queijo", "Tapioca"]

# ✏️ COMPLETE: troque os ___ pelo código certo
cardapio.append("Brigadeiro")          # adiciona no fim da lista
ultimo = cardapio[-1]              # pega o último item

print("Cardápio agora:", cardapio)
print("Último item:", ultimo)

conferir(len(cardapio) == 5 and ultimo == "Brigadeiro", "Isso! A lista tem 5 itens e o último é o Brigadeiro.", "a lista deveria ter 5 itens e terminar em Brigadeiro.")

Cardápio agora: ['Café expresso', 'Cappuccino', 'Pão de queijo', 'Tapioca', 'Brigadeiro']
Último item: Brigadeiro
✅ Isso! A lista tem 5 itens e o último é o Brigadeiro.


### 1.3 Tabelas com pandas
`pandas` é a biblioteca que cria e manipula tabelas. Cada **coluna** é como uma coluna do Excel, e você faz contas com a coluna inteira de uma vez, sem arrastar fórmula.

In [16]:
mini = pd.DataFrame({
    "produto": ["Café expresso", "Cappuccino", "Pão de queijo"],
    "preco": [6.00, 9.90, 5.50],
    "quantidade": [10, 4, 8],
})
mini

,produto,preco,quantidade
0,Café expresso,6.0,10
1,Cappuccino,9.9,4
2,Pão de queijo,5.5,8


🧠 **Pense:** crie a coluna `total` (preço vezes quantidade) e descubra o **faturamento** somando essa coluna.

In [17]:
# ✏️ COMPLETE: troque os ___ pelo código certo
mini["total"] = mini["preco"] * mini["quantidade"]      # conta feita na coluna inteira
faturamento = mini["total"].sum()                # dica: soma

print(mini)
print("Faturamento: R$", round(faturamento, 2))

conferir(round(faturamento, 2) == 143.6, "Faturamento de R$ 143,60. Você acabou de fazer o que uma tabela dinâmica faz.", "o faturamento deveria ser 143.6.")

         produto  preco  quantidade  total
0  Café expresso    6.0          10   60.0
1     Cappuccino    9.9           4   39.6
2  Pão de queijo    5.5           8   44.0
Faturamento: R$ 143.6
✅ Faturamento de R$ 143,60. Você acabou de fazer o que uma tabela dinâmica faz.


### 1.4 🐞 Seu primeiro erro (com a IA ao lado)
Todo mundo erra, e quem trabalha com dados também. A diferença é **saber ler o erro** e pedir ajuda bem. A célula abaixo **dá erro de propósito**. Rode e leia a mensagem em vermelho.

In [22]:
# 🐞 Esta célula tem um erro de propósito. Rode e leia a mensagem.
nome_produto = "Cappuccino"
print("Produto do dia:", nome_produto)

Produto do dia: Cappuccino


**Como ler um erro:** olhe a **última linha**. Ela diz o tipo do erro e o motivo. Aqui: `NameError: name 'nome_produo' is not defined` quer dizer que o Python não conhece essa palavra. Provavelmente é um erro de digitação.

Agora peça à IA para explicar e corrigir. Copie o prompt, troque o que está entre colchetes e cole no ChatGPT, Gemini ou Claude:

```
Rodei este código no Google Colab (Python) e deu erro.
Explique em linguagem simples o que o erro significa e mostre o código corrigido.

Código:
[cole o código da célula]

Erro:
[cole a mensagem de erro inteira]
```

✏️ **Corrija a célula acima e rode de novo.** Daqui para frente, sempre que aparecer um erro, esse é o caminho: ler, pedir ajuda à IA e conferir a correção.

---
# 🧺 Parte 2 · Coletar

**Coletar** é buscar os dados onde eles estão. O sistema de caixa do café exporta arquivos **CSV**: tabelas em texto simples, separadas por vírgula. É o formato mais comum de troca de dados, e o Excel também abre.

| `vendas.csv` | O que é |
|---|---|
| `id_venda` | código do pedido (um pedido pode ter vários itens) |
| `data_hora` | quando vendeu |
| `id_cliente` | quem comprou (pode estar vazio) |
| `id_produto` | código do produto (P01, P02...) |
| `quantidade` | quantas unidades |
| `preco_unitario` | preço cobrado por unidade |
| `canal` | Balcão ou iFood |
| `forma_pagamento` | Pix, cartão, dinheiro... |
| `tempo_entrega_min` | minutos até a entrega (só iFood) |

E o `produtos.csv` tem `id_produto`, `nome_produto`, `categoria`, `custo_unitario` e `preco_atual`.

In [23]:
vendas = ler("dados_brutos/vendas.csv")
produtos = ler("dados_brutos/produtos.csv")

print("Dados coletados! ✅")

Dados coletados! ✅


In [24]:
vendas = ler("dados_brutos/vendas.csv")
display(vendas.head())

,id_venda,data_hora,id_cliente,id_produto,quantidade,preco_unitario,canal,forma_pagamento,tempo_entrega_min
0,V005603,2026-03-07 14:52:06,sem cadastro,P15,1.0,8.00,Balcão,Pix,NaN
1,V005537,2026-03-06 09:19:30,C0019,P03,1.0,7.50,Balcão,Cartão de crédito,NaN
2,V015016,2026-06-24 13:11:24,NaN,P11,1.0,"13,00",Balcão,Cartão de crédito,NaN
3,V018999,2026-08-12 17:18:32,NaN,P08,1.0,12.00,balcao,Dinheiro,NaN
4,V005090,2026-03-01 16:40:35,C1014,p04,1.0,11.00,iFood,ifood online,17.0


🧠 **Pense:** o primeiro olhar em uma tabela responde duas perguntas: **quantas linhas e colunas tem?** e **como são as primeiras linhas?** Complete os comandos.

> 🤖 **Dica de IA:** *"Em pandas, como vejo o tamanho de um DataFrame e as primeiras linhas?"*

In [25]:
# ✏️ COMPLETE: troque os ___ pelo código certo
print("Linhas e colunas:", vendas.shape)      # dica: shape (sem parênteses)
display(vendas.head(8))                               # dica: head, para ver as 8 primeiras linhas

Linhas e colunas: (42868, 9)


,id_venda,data_hora,id_cliente,id_produto,quantidade,preco_unitario,canal,forma_pagamento,tempo_entrega_min
0,V005603,2026-03-07 14:52:06,sem cadastro,P15,1.0,8.00,Balcão,Pix,NaN
1,V005537,2026-03-06 09:19:30,C0019,P03,1.0,7.50,Balcão,Cartão de crédito,NaN
2,V015016,2026-06-24 13:11:24,NaN,P11,1.0,"13,00",Balcão,Cartão de crédito,NaN
3,V018999,2026-08-12 17:18:32,NaN,P08,1.0,12.00,balcao,Dinheiro,NaN
4,V005090,2026-03-01 16:40:35,C1014,p04,1.0,11.00,iFood,ifood online,17.0
5,V022195,2026-09-24 15:29:13,C0428,P15,1.0,8.00,Balcão,Pix,NaN
6,V010097,2026-04-29 12:43:47,C1169,P08,1.0,12.00,Balcão,Cartão de débito,NaN
7,V011657,2026-05-16 17:50:23,C0639,P02,1.0,9.90,Balcão,Pix,NaN


In [26]:
vendas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42868 entries, 0 to 42867
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_venda           42868 non-null  object 
 1   data_hora          42868 non-null  object 
 2   id_cliente         25429 non-null  object 
 3   id_produto         42868 non-null  object 
 4   quantidade         42699 non-null  float64
 5   preco_unitario     42868 non-null  object 
 6   canal              42868 non-null  object 
 7   forma_pagamento    42868 non-null  object 
 8   tempo_entrega_min  8998 non-null   float64
dtypes: float64(2), object(7)
memory usage: 2.9+ MB


🔎 **O que `info()` mostra:** o tipo de cada coluna. `float64` é número. `str` ou `object` é **texto**.

Repare que **`preco_unitario` e `data_hora` estão como texto**. Com preço em texto não dá para somar, e com data em texto não dá para agrupar por mês. Os dados chegaram **sujos**. Vamos limpar.

---
# 🧼 Parte 3 · Limpar

Dados do mundo real chegam com problemas: digitação diferente, formatos misturados, linhas repetidas. Quem trabalha com dados gasta boa parte do tempo aqui, e é uma etapa **decisiva**: dado sujo gera conclusão errada.

Nosso método em 3 passos: **🔎 Diagnosticar → 🔧 Corrigir → ✅ Conferir**.

Regra de ouro: **não mexa nos dados originais**. Vamos trabalhar em uma cópia chamada `limpas`.

### 🔎 Diagnóstico
Complete os dois comandos para investigar.

In [29]:
# ✏️ COMPLETE: troque os ___ pelo código certo
# 1) Quantas linhas estão duplicadas?       (dica: duplicated)

print("Linhas duplicadas:", vendas.duplicated().sum())

# 2) De quantas formas o canal está escrito?  (dica: value_counts)
vendas["canal"].value_counts()

Linhas duplicadas: 508


,count
canal,
Balcão,25417
iFood,6734
BALCAO,2851
balcao,2840
Balcao,2762
IFOOD,778
ifood,757
I-Food,729


In [30]:
# 3) Linhas de teste que alguém esqueceu no sistema
print("Linhas de TESTE:")
display(vendas[vendas["id_venda"].str.startswith("TESTE")])

# 4) Como estão escritos os preços e as datas?
vendas[["data_hora", "preco_unitario", "quantidade", "id_produto"]].sample(10, random_state=3)

Linhas de TESTE:


,id_venda,data_hora,id_cliente,id_produto,quantidade,preco_unitario,canal,forma_pagamento,tempo_entrega_min
11106,TESTE5,2026-02-10 06:00:00,NaN,P01,1.0,0,Balcão,Pix,NaN
19089,TESTE3,2026-02-10 06:00:00,NaN,P01,1.0,0,Balcão,Pix,NaN
21623,TESTE4,2026-02-10 06:00:00,NaN,P01,1.0,0,Balcão,Pix,NaN
27125,TESTE1,2026-02-10 06:00:00,NaN,P01,1.0,0,Balcão,Pix,NaN
28310,TESTE2,2026-02-10 06:00:00,NaN,P01,1.0,0,Balcão,Pix,NaN


,data_hora,preco_unitario,quantidade,id_produto
6299,04/09/2026 08:00,11.00,1.0,P04
14033,04-03-2026 09h39,6.50,1.0,P05
40204,2026-03-14 14:13:07,11.00,1.0,P04
6082,2026-02-23 13:33:26,12.00,1.0,P08
19040,2026-02-04 11:28:11,"9,90",1.0,P02
3439,2026-08-07 08:19:51,11.00,1.0,P04
12243,2026-07-22 13:42:01,11.00,1.0,P04
10949,2026-03-17 14:18:32,12.00,1.0,P08
5983,2026-06-21 16:33:47,"R$ 6,50",1.0,P05
41284,2026-01-03 18:20:12,6.00,1.0,P01


📝 **Lista de problemas encontrados** (cada um vira uma missão):

| Missão | Problema |
|---|---|
| **1** | Linhas **duplicadas** e linhas de **teste** |
| **2** | Código do produto em **minúscula** (`p04`) e canal escrito de **8 jeitos** |
| **3** | **Preço em texto**: `9.90`, `9,90`, `R$ 9,00` |
| **4** | **Data em texto**, em 3 formatos diferentes |
| **5** | **Quantidades impossíveis**: vazia, zero, negativa, 100 |

In [31]:
limpas = vendas.copy()      # trabalhamos na cópia
print("Linhas no início:", len(limpas))

Linhas no início: 42868


### 🧹 Missão 1 · Remover o que não deveria estar aqui
🧠 **Pense:** duplicadas saem com `drop_duplicates`. Já as linhas de teste são as que o `id_venda` **começa com** `TESTE`. O símbolo `~` significa **"não"**: ficamos com as linhas que *não* começam com TESTE.

> 🤖 **Dica de IA:** cole a segunda linha do código e pergunte *"Explique linha por linha, com um exemplo pequeno."* O `reset_index(drop=True)` só renumera as linhas depois da limpeza.

In [32]:
# ✏️ COMPLETE: troque os ___ pelo código certo
limpas = limpas.drop_duplicates().reset_index(drop=True)
limpas = limpas[~limpas["id_venda"].str.startswith("TESTE")].reset_index(drop=True)

print("Linhas agora:", len(limpas))
conferir(len(limpas) == 42355, "Perfeito! Sobraram 42.355 linhas.", "deveriam sobrar 42.355 linhas. Confira o comando das duplicadas e o texto TESTE.")

Linhas agora: 42355
✅ Perfeito! Sobraram 42.355 linhas.


### 🧹 Missão 2 · Padronizar os textos
- O código do produto vem `P04` e também `p04`. Vamos deixar tudo em **maiúscula** (`upper`).
- O canal vem de 8 jeitos. Um **dicionário** diz como cada jeito vira o valor certo (`"jeito errado": "jeito certo"`). Olhe de novo o resultado do `value_counts()` e complete os dois que faltam para o iFood.

In [33]:
# ✏️ COMPLETE: troque os ___ pelo código certo
limpas["id_produto"] = limpas["id_produto"].str.upper()      # dica: upper

padrao_canal = {
    "Balcão": "Balcão", "balcao": "Balcão", "BALCAO": "Balcão", "Balcao": "Balcão",
    "iFood": "iFood", "ifood": "iFood", "IFOOD": "iFood", "I-Food": "iFood",
}
limpas["canal"] = limpas["canal"].map(padrao_canal)

conferir(limpas["id_produto"].str.isupper().all(), "Códigos de produto em maiúscula.", "ainda há código em minúscula.")
conferir(limpas["canal"].notna().all() and set(limpas["canal"]) == {"Balcão", "iFood"}, "Canal padronizado: só Balcão e iFood.", "ainda há canal sem tradução (apareceu vazio). Veja os 8 jeitos no value_counts().")

✅ Códigos de produto em maiúscula.
✅ Canal padronizado: só Balcão e iFood.


### 🧹 Missão 3 · O preço virou número
O preço vem como `9.90`, `9,90` e `R$ 9,00`. Para virar número é preciso **tirar o "R$"**, **trocar a vírgula por ponto** e **converter**. Uma **função** (`def`) guarda esses passos para usarmos em todas as linhas, e o `apply` aplica a função em cada valor da coluna.

🧠 **Pense:** o que cada linha da função faz com o texto `"R$ 9,00"`? Complete os dois campos.

> 🤖 **Dica de IA:** cole a função e peça *"Mostre o que acontece, linha por linha, com o valor R$ 9,00."*

In [35]:
# ✏️ COMPLETE: troque os ___ pelo código certo
def limpar_preco(texto):
    texto = str(texto)
    texto = texto.replace("R$", "").strip()      # tira o "R$" e os espaços
    texto = texto.replace(",", ".")            # a vírgula decimal vira ponto
    return float(texto)

limpas["preco_unitario"] = limpas["preco_unitario"].apply(limpar_preco)      # dica: apply

conferir(limpas["preco_unitario"].dtype == "float64" and limpas["preco_unitario"].between(3, 20).all(), "Preço agora é número e os valores fazem sentido.", "o preço deveria ser número (float64), entre 3 e 20.")
limpas["preco_unitario"].describe()

✅ Preço agora é número e os valores fazem sentido.


,preco_unitario
count,42355.000000
mean,9.498258
std,3.094340
min,4.000000
25%,6.500000
50%,9.900000
75%,12.000000
max,16.000000


### 🤖 Missão 4 · As datas (a IA é a protagonista)
Aqui o trabalho é de **pedir bem à IA**. A data chegou em formatos diferentes. Olhe uma amostra:

In [36]:
limpas["data_hora"].sample(10, random_state=2)

,data_hora
32802,2026-07-30 07:04:15
21974,26/09/2026 12:38
6652,2026-09-14 15:47:02
27561,2026-07-15 14:54:50
18025,2026-06-18 16:47:57
26432,2026-06-03 10:32:21
30550,2026-03-28 15:19:46
2708,2026-01-31 09:29:20
30221,26/09/2026 12:03
15804,12/06/2026 07:48


🧠 **Pense:** você consegue identificar **3 formatos** diferentes? Escreva um exemplo de cada no lugar dos `___`. **Atenção:** em `05/07/2026` o primeiro número é o **dia** (5 de julho). Se você não avisar, a IA pode entender o contrário.

A célula abaixo monta o prompt completo, com as colunas e exemplos reais da sua tabela. **Copie todo o texto que aparecer, cole na IA** e leia a resposta antes de copiar o código.

In [37]:
# ✏️ COMPLETE: troque os ___ pelo código certo
pedido = """Preciso transformar a coluna `data_hora` em datas de verdade (datetime).
Hoje ela é texto e aparece em 3 formatos:
  - 2026-07-30 07:04:15
  - 26/09/2026 12:38   (dia/mês/ano)
  - 12-06-2026 07h48
O resultado deve ficar na própria coluna `data_hora` da tabela `limpas`.
Os dados são de 2026."""

print(prompt_com_amostra(limpas, "limpas", pedido))

Sou iniciante em análise de dados e uso Python (pandas) no Google Colab.

Minha tabela se chama `limpas` e tem estas colunas:
id_venda, data_hora, id_cliente, id_produto, quantidade, preco_unitario, canal, forma_pagamento, tempo_entrega_min

Algumas linhas de exemplo (dados fictícios):
id_venda           data_hora id_cliente id_produto  quantidade  preco_unitario  canal forma_pagamento  tempo_entrega_min
 V000408 2026-01-05 14:39:28        NaN        P05         1.0             6.5  iFood  iFood (online)               26.0
 V022347    26/09/2026 17:34        NaN        P02         2.0            12.9  iFood  iFood (online)               42.0
 V017559 2026-07-24 17:07:46      C1176        P02         1.0            12.9 Balcão        Dinheiro                NaN
 V002252 2026-01-25 07:18:01      C0542        P14         1.0            10.0 Balcão             Pix                NaN
 V019378 2026-08-17 18:15:20      C0137        P11         1.0            13.0 Balcão             Pix       

Agora **cole o código que a IA devolveu** na célula abaixo e rode. Ele precisa deixar o resultado em `limpas["data_hora"]`.

> 🤖 Se a IA devolveu um erro ou um resultado estranho, use o prompt de **corrigir erro** (da Parte 1.4). Se a IA entregou várias opções, escolha a mais simples e peça "explique linha por linha".
> 🆘 Não funcionou mesmo? Veja o **Apêndice A** no final do notebook.

In [38]:
# 🤖 COLE AQUI o código que a IA escreveu
# (ele deve atualizar a coluna limpas["data_hora"])
def converter_data(coluna):
    formatos = ["%Y-%m-%d %H:%M:%S", "%%d/%m/%Y %H:%M", "%%d-%%m-%%Y %Hh%%M"]
    resultado = pd.Series(pd.NaT, index=coluna.index)
    for formato in formatos:
        convertido = pd.to_datetime(coluna, format=formato, errors="coerce")
        resultado = resultado.fillna(convertido)
    return resultado

limpas["data_hora"] = converter_data(limpas["data_hora"])

In [39]:
# ✅ Conferência da Missão 4 (não precisa mexer)
e_data = pd.api.types.is_datetime64_any_dtype(limpas["data_hora"])
conferir(e_data, "A coluna data_hora virou data de verdade.", "a coluna ainda é texto. Cole o código da IA na célula acima e rode.")

if e_data:
    vazias = limpas["data_hora"].isna().sum()
    conferir(vazias == 0, "Nenhuma data ficou vazia.", f"{vazias} datas ficaram vazias: algum formato não foi reconhecido. Mostre esses exemplos à IA.")

    if vazias == 0:
        # Validação cruzada: o código da venda (V000001, V000002...) cresce com o tempo.
        # Se dia e mês estiverem trocados, as datas ficam fora de ordem.
        ordem = limpas["id_venda"].str[1:].astype(int).corr(limpas["data_hora"].astype("int64"))
        conferir(ordem > 0.99, "As datas fazem sentido: dia e mês estão na ordem certa.", "dia e mês parecem trocados em algum formato. Avise a IA que o formato com barras é dia/mês/ano.")

✅ A coluna data_hora virou data de verdade.
❌ Ainda não: 12713 datas ficaram vazias: algum formato não foi reconhecido. Mostre esses exemplos à IA.


> 💡 **Aprendizado:** olha o que acabamos de fazer na célula acima. Conferimos as datas usando **outra informação** (o código da venda cresce com o tempo). Em dados, **validar com uma segunda fonte** é o que separa um resultado confiável de um palpite.

### 🧹 Missão 5 · Quantidades impossíveis
A quantidade tem valores vazios, `0`, `-1` e até `100` (alguém digitou errado). Uma cafeteria não vende 100 cafés em um item. Vamos manter só quantidades de **1 a 10**.

🧠 **Pense:** `pd.to_numeric(..., errors="coerce")` transforma em número e o que não for número vira vazio. O `between(1, 10)` separa as linhas válidas.

In [40]:
# ✏️ COMPLETE: troque os ___ pelo código certo
limpas["quantidade"] = pd.to_numeric(limpas["quantidade"], errors="coerce")

limpas = limpas[limpas["quantidade"].between(1, 10)].reset_index(drop=True)      # só de 1 a 10
limpas["quantidade"] = limpas["quantidade"].astype(int)

print(f"Linhas: {len(vendas):,} no início → {len(limpas):,} depois da limpeza".replace(",", "."))
conferir(len(limpas) == 42101, "Bateu com o esperado: 42.101 linhas. Limpeza concluída! 🎉", "deveriam sobrar 42.101 linhas. Confira o limite máximo da quantidade (10).")

Linhas: 42.868 no início → 42.101 depois da limpeza
✅ Bateu com o esperado: 42.101 linhas. Limpeza concluída! 🎉


🎯 **O que fizemos:** diagnosticamos, corrigimos 5 problemas e conferimos cada um. Sem isso, qualquer gráfico ou conclusão estaria errado.

> ⏸️ Ficou para trás na limpeza? Sem problema: o **Apêndice B** (no final) carrega os dados já limpos para você acompanhar a próxima parte.

---
# 🔧 Parte 4 · Transformar

**Transformar** é dar aos dados o formato que a análise precisa: **juntar tabelas** e **criar colunas novas**.

### 4.1 Juntar tabelas
As vendas só têm o **código** do produto (`P02`). O **nome** (Cappuccino) e a **categoria** estão na tabela `produtos`. O `merge` junta as duas usando a coluna que existe nas duas: é o famoso **PROCV** do Excel.

> 🤖 **Dica de IA:** *"Em pandas, como junto duas tabelas usando uma coluna em comum, mantendo todas as linhas da primeira? Equivalente ao PROCV."*

In [41]:
vendas_prontas = limpas.merge(
    produtos[["id_produto", "nome_produto", "categoria"]],
    on="id_produto",          # a coluna que existe nas duas tabelas
    how="left",
)

conferir(vendas_prontas["nome_produto"].notna().all(), "Todos os produtos foram encontrados.", "alguns produtos ficaram sem nome. Confira a coluna do merge e a Missão 2 (maiúsculas).")
vendas_prontas.head()

✅ Todos os produtos foram encontrados.


,id_venda,data_hora,id_cliente,id_produto,quantidade,preco_unitario,canal,forma_pagamento,tempo_entrega_min,nome_produto,categoria
0,V005603,2026-03-07 14:52:06,sem cadastro,P15,1,8.0,Balcão,Pix,NaN,Bolo de macaxeira,Doce
1,V005537,2026-03-06 09:19:30,C0019,P03,1,7.5,Balcão,Cartão de crédito,NaN,Café com leite,Bebida quente
2,V015016,2026-06-24 13:11:24,NaN,P11,1,13.0,Balcão,Cartão de crédito,NaN,Cuscuz com ovo,Salgado
3,V018999,2026-08-12 17:18:32,NaN,P08,1,12.0,Balcão,Dinheiro,NaN,Tapioca de queijo coalho,Salgado
4,V005090,2026-03-01 16:40:35,C1014,P04,1,11.0,iFood,ifood online,17.0,Latte,Bebida quente


### 4.2 Criar colunas novas
🧠 **Pense:** quanto cada linha faturou? E em que **mês** e **hora** ela aconteceu? Como a data agora é "de verdade", o Python extrai o mês e a hora com `.dt`.

In [45]:
vendas_prontas["valor_total"] = vendas_prontas["quantidade"] * vendas_prontas["preco_unitario"]      # quantidade x preço

vendas_prontas["mes"] = vendas_prontas["data_hora"].dt.month      # dica: month
vendas_prontas["hora"] = vendas_prontas["data_hora"].dt.hour

vendas_prontas[["data_hora", "quantidade", "preco_unitario", "valor_total", "mes", "hora"]].head()

,data_hora,quantidade,preco_unitario,valor_total,mes,hora
0,2026-03-07 14:52:06,1,8.0,8.0,3.0,14.0
1,2026-03-06 09:19:30,1,7.5,7.5,3.0,9.0
2,2026-06-24 13:11:24,1,13.0,13.0,6.0,13.0
3,2026-08-12 17:18:32,1,12.0,12.0,8.0,17.0
4,2026-03-01 16:40:35,1,11.0,11.0,3.0,16.0


### 4.3 Uma regra de negócio: período do dia
Vamos classificar cada venda em **Manhã**, **Almoço**, **Tarde** ou **Noite**. Isso se escreve com uma **função** e `if / elif / else` ("se / senão se / senão").

🧠 **Pense:** manhã vai até as 10h, a tarde começa depois das 14h e vai até as 17h. Qual é o limite do **Almoço**?

In [46]:
def periodo_do_dia(hora):
    if hora <= 10:
        return "Manhã"
    elif hora <= 14:
        return "Almoço"
    elif hora <= 17:
        return "Tarde"
    else:
        return "Noite"

vendas_prontas["periodo"] = vendas_prontas["hora"].apply(periodo_do_dia)

conferir(set(vendas_prontas["periodo"]) == {"Manhã", "Almoço", "Tarde", "Noite"}, "Os quatro períodos foram criados.", "deveriam aparecer Manhã, Almoço, Tarde e Noite.")
vendas_prontas["periodo"].value_counts()

✅ Os quatro períodos foram criados.


,count
periodo,
Noite,15261
Manhã,12066
Tarde,8048
Almoço,6726


In [ ]:
# ✅ Conferência final da Parte 4
total = vendas_prontas["valor_total"].sum()
print(f"Faturamento total: R$ {total:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
conferir(round(total) == 459089, "Bateu com o esperado! Dados prontos para examinar. 🎉", "o faturamento total deveria ser R$ 459.089. Revise as Missões 3 e 5 e a coluna valor_total.")

---
# 🔍 Parte 5 · Examinar com SQL

Os dados estão limpos e prontos. Agora vamos **fazer perguntas** a eles. Para isso, usamos o **SQL**, a linguagem de perguntas a tabelas, que aparece em quase toda vaga de dados.

Se você já fez **tabela dinâmica** no Excel, vai reconhecer: `GROUP BY` agrupa, `SUM` soma, `WHERE` filtra.

```sql
SELECT mes, SUM(valor_total)     -- o que eu quero ver (colunas e contas)
FROM vendas_prontas              -- de qual tabela
WHERE canal = 'iFood'            -- (opcional) filtro
GROUP BY mes                     -- (opcional) agrupar
ORDER BY mes                     -- (opcional) ordenar
LIMIT 5                          -- (opcional) só as 5 primeiras linhas
```

| Função | O que faz |
|---|---|
| `SUM(coluna)` | soma |
| `COUNT(DISTINCT coluna)` | conta valores diferentes (ex.: pedidos) |
| `AVG(coluna)` | média |
| `ROUND(valor, 2)` | arredonda para 2 casas |
| `... AS nome` | dá um apelido à coluna do resultado |

> 🤖 **Dica de IA:** quando não souber escrever uma consulta, descreva a pergunta em português. Mais adiante você vai ver como a IA escreve o SQL por você.

Primeiro, um olhar para a tabela:

In [ ]:
sql("SELECT * FROM vendas_prontas LIMIT 5")

### Pergunta 1 · As vendas caíram mesmo? Faturamento por mês
🧠 **Pense:** queremos o faturamento de **cada mês**. Em que coluna agrupar?

✍️ *Antes de rodar, escreva o seu palpite: em qual mês você acha que o faturamento começou a cair?*

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
fat_mes = sql("""
    SELECT mes, ROUND(SUM(valor_total), 2) AS faturamento
    FROM vendas_prontas
    GROUP BY ___
    ORDER BY mes
""")
fat_mes

In [ ]:
fat_mes.plot(x="mes", y="faturamento", kind="bar", legend=False, color="#1FD3F0", title="Faturamento por mês (R$)")
plt.xlabel("mês")
plt.show()

✍️ **O que você viu?** Quando o faturamento caiu? Houve algum mês estranho? (Dica: julho é mês de férias em João Pessoa, com mais turistas.)

---
### Pergunta 2 · Quais produtos mais vendem?
Para ver um **ranking**, ordenamos de forma **decrescente** (`DESC`) e limitamos a quantidade de linhas (`LIMIT`).

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
top_produtos = sql("""
    SELECT nome_produto, SUM(quantidade) AS unidades
    FROM vendas_prontas
    GROUP BY nome_produto
    ORDER BY unidades ___
    LIMIT ___
""")
top_produtos

O **Cappuccino** é o campeão de vendas da casa. Então vale perguntar: o que aconteceu com ele ao longo dos meses?

---
### Pergunta 3 · E o Cappuccino?
Agora filtramos só o produto que queremos com `WHERE`. Texto no SQL vai **entre aspas simples**.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
capp_mes = sql("""
    SELECT mes,
           ROUND(AVG(preco_unitario), 2) AS preco_medio,
           SUM(quantidade) AS unidades
    FROM vendas_prontas
    WHERE nome_produto = '___'
    GROUP BY mes
    ORDER BY mes
""")
capp_mes

✍️ **O que aconteceu com o preço? E com as unidades vendidas?** Compare junho com os meses seguintes.

---
### Pergunta 4 · E o delivery (iFood)?
Quantos pedidos o iFood trouxe por mês e quanto tempo demorou a entrega?

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
ifood_mes = sql("""
    SELECT mes,
           COUNT(DISTINCT id_venda) AS pedidos_ifood,
           ROUND(AVG(tempo_entrega_min), 1) AS tempo_medio_entrega
    FROM vendas_prontas
    WHERE canal = '___'
    GROUP BY mes
    ORDER BY mes
""")
ifood_mes

In [ ]:
ifood_mes.plot(x="mes", y="tempo_medio_entrega", kind="line", marker="o", legend=False, color="#3DF29A", title="Tempo médio de entrega no iFood (minutos)")
plt.xlabel("mês")
plt.show()

✍️ **O que o gráfico mostra?** Em que mês a entrega piorou? Isso pode ter relação com os pedidos que caíram?

---
### Pergunta 5 · 🤖 Agora a IA escreve o SQL (desafio)
Em qual **período do dia** as vendas mais caíram? Compare **junho** (antes dos problemas) com **setembro**.

Em vez de escrever o SQL, você vai **pedir à IA**. A célula abaixo monta o prompt com as colunas e exemplos da sua tabela. Copie, cole na IA, **leia a explicação** e traga o SQL de volta.

In [ ]:
pergunta = "Quantos pedidos (id_venda diferentes) houve em cada período do dia, em junho e em setembro? Quero ver os dois meses lado a lado."

print(prompt_sql(vendas_prontas, "vendas_prontas", pergunta))

In [ ]:
sql("""
-- 🤖 COLE AQUI o SQL que a IA escreveu (apague esta linha)
""")

✍️ **O que o resultado mostra?** Qual período perdeu mais pedidos? O que isso sugere sobre o que mudou na vizinhança do café?

> 🤖 **Dica de IA (conferir é obrigatório):** peça à própria IA para revisar o SQL dela: *"Esse SQL responde exatamente à pergunta? Existe risco de contar a mesma venda duas vezes?"* E sempre confira um número "na mão" (por exemplo, o total de junho nesta consulta contra o da Pergunta 1).

---
# 💡 Parte 6 · Insights e ✅ Decisão

Dados e tabelas não convencem ninguém sozinhos. **Insight** é um achado com **número**, **evidência** e um **"e daí?"**:

> *"O Cappuccino subiu de R$ 9,90 para R$ 12,90 e as unidades vendidas caíram mais da metade. É o produto campeão, então o preço pode ter afastado clientes."*

Esta célula reúne as evidências que você acabou de descobrir (junho = antes dos problemas, setembro = depois):

In [ ]:
# 📋 FICHA DE EVIDÊNCIAS (não precisa mexer, só rode)
def valor(tabela, mes, coluna):
    return float(tabela.loc[tabela["mes"] == mes, coluna].iloc[0])

def brl(x):
    return f"R$ {x:,.0f}".replace(",", ".")

f6, f9 = valor(fat_mes, 6, "faturamento"), valor(fat_mes, 9, "faturamento")
p6, p9 = valor(capp_mes, 6, "preco_medio"), valor(capp_mes, 9, "preco_medio")
u6, u9 = valor(capp_mes, 6, "unidades"), valor(capp_mes, 9, "unidades")
i6, i9 = valor(ifood_mes, 6, "pedidos_ifood"), valor(ifood_mes, 9, "pedidos_ifood")
t6, t9 = valor(ifood_mes, 6, "tempo_medio_entrega"), valor(ifood_mes, 9, "tempo_medio_entrega")

print("📋 FICHA DE EVIDÊNCIAS (junho → setembro)\n")
print(f"💰 Faturamento: {brl(f6)} → {brl(f9)} ({f9/f6-1:+.0%})")
print(f"☕ Cappuccino: preço R$ {p6:.2f} → R$ {p9:.2f} | unidades {u6:.0f} → {u9:.0f} ({u9/u6-1:+.0%})")
print(f"🛵 iFood: pedidos {i6:.0f} → {i9:.0f} ({i9/i6-1:+.0%}) | entrega {t6:.0f} min → {t9:.0f} min")

### Seus achados
🧠 **Pense:** escreva **3 achados**. Cada um com um **número** da ficha acima e o que ele significa. Depois a célula monta um prompt para a IA ajudar a **contar a história ao dono do café**.

> 🛡️ **Atenção ao usar IA aqui:** o prompt já pede para a IA usar **somente os seus números**. Mesmo assim, **confira cada número** no texto que ela escrever. IA escreve bonito, mas quem garante que é verdade é você.

In [ ]:
# ✏️ COMPLETE: troque os ___ pelo código certo
meus_achados = """
1) ___
2) ___
3) ___
"""

print(prompt_insights(meus_achados))

### ✅ Decisão
O que o dono do Café Tambaú deveria fazer? Escreva **3 recomendações**, da mais urgente para a menos urgente. Cada uma precisa se apoiar em um achado.

✍️ *Clique duas vezes aqui e escreva:*

> 1. ...
> 2. ...
> 3. ...

### 🎯 Conferindo o palpite
Volte ao palpite do início do notebook. Quem acertou? A IA te ajudou a **escrever** código, mas foi o seu **raciocínio** que ligou preço, entrega e vendas.

---
## 🎓 O que você praticou hoje

| Etapa | O que você fez | Comandos |
|---|---|---|
| 🧺 **Coletar** | leu arquivos CSV | `pd.read_csv`, `head`, `shape`, `info` |
| 🧼 **Limpar** | achou e corrigiu 5 problemas | `duplicated`, `drop_duplicates`, `str.upper`, `map`, `apply`, `to_numeric` |
| 🔧 **Transformar** | juntou tabelas e criou colunas | `merge`, `dt.month`, `def` e `if` |
| 🔍 **Examinar** | fez perguntas com SQL | `SELECT`, `WHERE`, `GROUP BY`, `ORDER BY`, `LIMIT` |
| 💡 **Insights** | contou a história com números | evidências + decisão |
| 🤖 **IA** | pediu código, corrigiu erro, escreveu SQL e resumiu | prompt com contexto, dados e formato |

### 🗺️ Onde isso aparece no Amazing Data IA Lab
Hoje você viu um pouco de cada parte. No curso, cada uma ganha tempo e profundidade:

| Você viu hoje | Vai aprofundar em |
|---|---|
| Python e variáveis | Aula 1 · Fundamentos de Python |
| pandas, limpeza e transformação | Aula 2 · Pandas: manipulação de dados |
| gráficos e padrões | Aula 3 · Análise exploratória e visualização |
| SQL (`SELECT`, `GROUP BY`) | Aula 4 · Fundamentos de SQL |
| juntar tabelas | Aula 5 · SQL aplicado e JOINs |
| IA para código e para insights | Aulas 6 e 7 · IA para produtividade e para insights de negócio |
| contar a história | Aulas 8 e 9 · Dashboards e storytelling com dados |
| tudo junto | Aula 10 · Projeto final |

**Nerd Data Lab × UNIESP Centro Universitário** · *Dados + Pessoas = Melhores Decisões.*

---
## 📎 Colinha

**Python e pandas**
```python
tabela = pd.read_csv("arquivo.csv")        # ler um CSV
tabela.shape / tabela.head() / tabela.info()
tabela["coluna"].value_counts()             # contar valores
tabela.duplicated().sum()                   # quantas linhas repetidas
tabela.drop_duplicates()                    # remover repetidas
tabela["coluna"].str.upper()                # texto em maiúscula
tabela["coluna"].map({"velho": "novo"})     # trocar valores
tabela["coluna"].apply(minha_funcao)        # aplicar uma função em cada valor
pd.to_numeric(tabela["coluna"], errors="coerce")
tabela[tabela["coluna"] > 10]               # filtrar linhas
a.merge(b, on="codigo", how="left")         # juntar tabelas (PROCV)
tabela["data"].dt.month                     # mês de uma data
```

**SQL**
```sql
SELECT coluna, SUM(valor) AS total
FROM tabela
WHERE canal = 'iFood'
GROUP BY coluna
ORDER BY total DESC
LIMIT 5
```

**Prompts para guardar**
```
EXPLICAR:  "Sou iniciante. Explique linha por linha, em português simples, com um exemplo pequeno: [código]"
ERRO:      "Rodei este código no Colab e deu erro. Explique e corrija. Código: [código] Erro: [erro]"
REVISAR:   "Revise este código/SQL: ele responde exatamente à pergunta '[pergunta]'? Que erros comuns posso ter cometido?"
```

---
# 🆘 Apêndice · Saídas de emergência

Use **só depois de tentar**. Aprender é tentar, errar, pedir ajuda e acertar.

## Apêndice A · Socorro da Missão 4 (datas)
Rode a célula abaixo. Ela cria a função `converter_data`. Depois **volte à célula "COLE AQUI"** da Missão 4, escreva a linha abaixo e rode a conferência.

```python
limpas["data_hora"] = converter_data(limpas["data_hora"])
```

In [ ]:
# 🆘 Apêndice A: função que converte os 3 formatos de data
def converter_data(coluna):
    formatos = ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%d-%m-%Y %Hh%M"]
    resultado = pd.Series(pd.NaT, index=coluna.index)
    for formato in formatos:
        convertido = pd.to_datetime(coluna, format=formato, errors="coerce")
        resultado = resultado.fillna(convertido)
    return resultado

print("Função criada. Agora use: limpas[\"data_hora\"] = converter_data(limpas[\"data_hora\"])")

## Apêndice B · Retomada: continuar com os dados já limpos
Ficou para trás na limpeza ou quer tirar a prova? Esta célula carrega a versão **já limpa e pronta** da tabela e cria a `vendas_prontas`, para você seguir direto para a **Parte 5**. (A tabela do instrutor tem as mesmas colunas, mas o número de linhas pode diferir um pouco da sua, porque os critérios de limpeza não são idênticos.)

In [ ]:
# 🆘 Apêndice B: carrega a tabela já limpa e pronta para a Parte 5
vendas_prontas = ler("dados_tratados/vendas_limpas.csv", parse_dates=["data_hora"])
print(f"Tabela pronta carregada: {len(vendas_prontas):,} linhas ✅".replace(",", "."))
print("Agora rode de novo as perguntas da Parte 5, a partir de 'SELECT * FROM vendas_prontas'.")